#### Setup

In [ ]:
!pip install -q langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers transformers accelerate

#### Sample document

In [14]:
import ipywidgets as widgets
from IPython.display import display

uploader = widgets.FileUpload(
    accept=".txt",
    multiple=False
)
display(uploader)

FileUpload(value={}, accept='.txt', description='Upload')

In [15]:
uploaded_file = list(uploader.value.values())[0]

file_name = uploaded_file["metadata"]["name"]
file_content = uploaded_file["content"].decode("utf-8")

with open("data.txt", "w", encoding="utf-8") as f:
    f.write(file_content)

print("Uploaded file:", file_name)
print("Saved as data.txt")
print("\nPreview:")
print(file_content[:500])

Uploaded file: data.txt
Saved as data.txt

Preview:
Climate change refers to long-term changes in Earth's temperature, weather patterns, oceans, and ecosystems, and it has become one of the most significant environmental, economic, and social challenges of the modern era. Although Earth's climate has changed naturally throughout history due to factors such as volcanic activity, changes in solar radiation, and variations in the planet's orbit, the rapid warming observed since the Industrial Revolution is primarily associated with human activities.


#### Load and chunk document

In [17]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("data.txt")
documents = loader.load()
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=30
)
chunks = splitter.split_documents(documents)

print("Total chunks:", len(chunks))

Total chunks: 88


In [18]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vector_db = FAISS.from_documents(chunks, embedding_model)
retriever = vector_db.as_retriever(
    search_kwargs={"k": 2}
)
print("Vector database created successfully")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector database created successfully


#### Qwen model load

In [19]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline

model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float32,
    device_map="cpu"
)
text_generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=80,
    temperature=0.2,
    do_sample=False,
    return_full_text=False
)
llm = HuggingFacePipeline(
    pipeline=text_generation_pipeline
)
print("Model loaded")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded


#### RAG prompt

In [21]:
from langchain_core.prompts import PromptTemplate
prompt = PromptTemplate.from_template("""
You are a helpful assistant.

Answer the question using only the context below.
If the answer is not present in the context, say:
"I don't know based on the given context."

Context:
{context}

Question:
{question}

Answer:
""")

#### RAG function

In [22]:
def ask_rag(question):
    retrieved_docs = retriever.invoke(question)

    context = "\n\n".join(
        doc.page_content for doc in retrieved_docs
    )

    final_prompt = prompt.format(
        context=context,
        question=question
    )

    answer = llm.invoke(final_prompt)

    print("QUESTION:")
    print(question)

    print("\nRETRIEVED CONTEXT:")
    print(context)

    print("\nANSWER:")
    print(answer)

In [26]:
ask_rag("How can climate change affect both human health and agriculture?")

[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION:
How can climate change affect both human health and agriculture?

RETRIEVED CONTEXT:
reduce the amount of water available for irrigation. Climate change can also influence the spread of agricultural pests, weeds, and plant diseases. Farmers may respond by using drought-resistant

Natural ecosystems are also affected by a changing climate. Plants and animals are adapted to particular environmental conditions, and rapid shifts in temperature or rainfall can disrupt habitats,

ANSWER:
Climate change can have both direct and indirect effects on human health and agriculture. Directly, it can lead to increased frequency and severity of natural disasters such as hurricanes, floods, and wildfires. These events can cause significant damage to infrastructure, crops, and livestock, leading to loss of life and economic impact. Additionally, extreme weather patterns can trigger heatwaves, droughts, and other climatic events that
